In [1]:
import os
os.chdir('/home/smallyan/eval_agent')
print(f"Working directory: {os.getcwd()}")

Working directory: /home/smallyan/eval_agent


# Circuit Analysis Code Evaluation

This notebook evaluates the code implementing circuit analysis in `/net/scratch2/smallyan/leela_eval`.

## Setup and Initial Exploration

In [2]:
# Check GPU availability and inherit environment
import os
import torch

# Source bashrc to get environment variables
bashrc_path = os.path.expanduser("~/.bashrc")
if os.path.exists(bashrc_path):
    # Parse bashrc for export statements
    with open(bashrc_path, 'r') as f:
        for line in f:
            line = line.strip()
            if line.startswith('export '):
                parts = line[7:].split('=', 1)
                if len(parts) == 2:
                    key = parts[0]
                    value = parts[1].strip('"\'')
                    os.environ[key] = value

print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")
print(f"HF_HOME: {os.environ.get('HF_HOME', 'Not set')}")

CUDA available: True
CUDA device: NVIDIA A100 80GB PCIe
HF_HOME: /net/projects2/chai-lab/shared_models


## Project Structure

Based on the CodeWalkthrough.md, the main analysis files are:

**Notebooks:**
1. `notebooks/demo.ipynb` - Demo/quickstart showing core functionalities
2. `notebooks/figure1.ipynb` - Main figure plotting
3. `notebooks/forgotten_puzzle_figure.ipynb` - Forgotten puzzle analysis figure
4. `notebooks/policy_metrics.ipynb` - Policy distribution metrics (entropy, JS-divergence)
5. `notebooks/puzzle_results.ipynb` - Puzzle solving results analysis
6. `notebooks/tournament_results.ipynb` - Tournament Elo results analysis

**Scripts:**
1. `scripts/evaluate_puzzles.py` - Puzzle evaluation script
2. `scripts/evaluate_concepts.py` - Concept evaluation script  
3. `scripts/tournament.py` - Tournament script

Let me evaluate each of these systematically.

## 1. Evaluating demo.ipynb

Let me run through the demo.ipynb notebook code cell by cell to evaluate runnability and correctness.

In [3]:
# First, make sure we're in the right directory and install the package
import sys
sys.path.insert(0, '/net/scratch2/smallyan/leela_eval/src')
os.chdir('/net/scratch2/smallyan/leela_eval')
print(f"Working directory: {os.getcwd()}")

Working directory: /net/scratch2/smallyan/leela_eval


In [4]:
# Initialize a data structure for tracking evaluation results
evaluation_results = []

# Helper function to record results
def record_result(file_name, cell_id, runnable, correct, redundant, irrelevant, error_note=""):
    evaluation_results.append({
        "file": file_name,
        "cell_id": cell_id,
        "runnable": runnable,
        "correct_impl": correct,
        "redundant": redundant,
        "irrelevant": irrelevant,
        "error_note": error_note
    })
    
corrected_blocks = 0
blocks_that_failed = 0

In [5]:
# demo.ipynb Cell 1 - Import leela_interp
try:
    from leela_interp import Lc0sight, LeelaBoard
    record_result("demo.ipynb", "cell_1", "Y", "Y", "N", "N")
    print("Cell 1: SUCCESS - Imports work")
except Exception as e:
    record_result("demo.ipynb", "cell_1", "N", "N", "N", "N", str(e))
    print(f"Cell 1: FAILED - {e}")

Cell 1: SUCCESS - Imports work


In [6]:
# demo.ipynb Cell 2 - Set device (modified to use GPU as per instructions)
try:
    device = "cuda" if torch.cuda.is_available() else "cpu"
    record_result("demo.ipynb", "cell_2", "Y", "Y", "N", "N")
    print(f"Cell 2: SUCCESS - Device set to {device}")
except Exception as e:
    record_result("demo.ipynb", "cell_2", "N", "N", "N", "N", str(e))
    print(f"Cell 2: FAILED - {e}")

Cell 2: SUCCESS - Device set to cuda


In [7]:
# demo.ipynb Cell 3 - Load model
try:
    model = Lc0sight("lc0-original.onnx", device=device)
    record_result("demo.ipynb", "cell_3", "Y", "Y", "N", "N")
    print("Cell 3: SUCCESS - Model loaded")
except Exception as e:
    record_result("demo.ipynb", "cell_3", "N", "N", "N", "N", str(e))
    print(f"Cell 3: FAILED - {e}")

Using device: cuda


Cell 3: FAILED - Lc0Model(
  (_lc0_model): GraphModule(
    (attn_body/transpose): OnnxTranspose()
    (initializers): Module()
    (attn_body/reshape): OnnxReshape()
    (attn_body/shape): OnnxShape()
    (attn_body/batch): OnnxSlice()
    (attn_body/pos_encoding_shape): OnnxConcat()
    (attn_body/expand): OnnxExpand()
    (attn_body/padded_input): OnnxConcat()
    (attn_body/reshape2): OnnxReshape()
    (attn_body/matmul): OnnxMatMul()
    (attn_body/add): OnnxBinaryMathOperation()
    (attn_body/mish/softplus): Softplus(beta=1.0, threshold=20.0)
    (attn_body/mish/tanh): OnnxFunction()
    (attn_body/mish): OnnxBinaryMathOperation()
    (attn_body/ma_gating/rehape1): OnnxReshape()
    (ip_mul_gate): OnnxBinaryMathOperation()
    (ip_add_gate): OnnxBinaryMathOperation()
    (attn_body/ma_gating/rehape2): OnnxReshape()
    (encoder0/mha/Q/w): OnnxMatMul()
    (encoder0/mha/Q/b): OnnxBinaryMathOperation()
    (encoder0/mha/Q/reshape): OnnxReshape()
    (encoder0/mha/Q/transpose): Onn

In [8]:
# Check model actually works - the "error" was just that the model printed its architecture
# Let me try again more carefully
try:
    model = Lc0sight("lc0-original.onnx", device=device)
    # If the model object exists, it loaded successfully
    print(f"Model type: {type(model)}")
    print("Cell 3: SUCCESS - Model loaded on", device)
    # Remove failed record and add success
    evaluation_results = [r for r in evaluation_results if r['cell_id'] != 'cell_3']
    record_result("demo.ipynb", "cell_3", "Y", "Y", "N", "N")
except Exception as e:
    print(f"Cell 3: FAILED - {str(e)[:200]}")

Using device: cuda


Cell 3: FAILED - Lc0Model(
  (_lc0_model): GraphModule(
    (attn_body/transpose): OnnxTranspose()
    (initializers): Module()
    (attn_body/reshape): OnnxReshape()
    (attn_body/shape): OnnxShape()
    (attn_body/


In [9]:
import traceback

try:
    model = Lc0sight("lc0-original.onnx", device=device)
    print("Model loaded successfully")
    print(f"Model type: {type(model)}")
except Exception as e:
    print(f"ERROR TYPE: {type(e)}")
    print(f"ERROR: {e}")
    traceback.print_exc()

Using device: cuda


ERROR TYPE: <class 'AttributeError'>
ERROR: Lc0Model(
  (_lc0_model): GraphModule(
    (attn_body/transpose): OnnxTranspose()
    (initializers): Module()
    (attn_body/reshape): OnnxReshape()
    (attn_body/shape): OnnxShape()
    (attn_body/batch): OnnxSlice()
    (attn_body/pos_encoding_shape): OnnxConcat()
    (attn_body/expand): OnnxExpand()
    (attn_body/padded_input): OnnxConcat()
    (attn_body/reshape2): OnnxReshape()
    (attn_body/matmul): OnnxMatMul()
    (attn_body/add): OnnxBinaryMathOperation()
    (attn_body/mish/softplus): Softplus(beta=1.0, threshold=20.0)
    (attn_body/mish/tanh): OnnxFunction()
    (attn_body/mish): OnnxBinaryMathOperation()
    (attn_body/ma_gating/rehape1): OnnxReshape()
    (ip_mul_gate): OnnxBinaryMathOperation()
    (ip_add_gate): OnnxBinaryMathOperation()
    (attn_body/ma_gating/rehape2): OnnxReshape()
    (encoder0/mha/Q/w): OnnxMatMul()
    (encoder0/mha/Q/b): OnnxBinaryMathOperation()
    (encoder0/mha/Q/reshape): OnnxReshape()
    (enc

Traceback (most recent call last):
  File "/tmp/ipykernel_2213493/3551670248.py", line 4, in <module>
    model = Lc0sight("lc0-original.onnx", device=device)
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/smallyan/.conda/envs/meta/lib/python3.11/site-packages/leela_interp/core/nnsight.py", line 36, in __init__
    self._envoy._lc0_model, name
    ^^^^^^^^^^^
  File "/home/smallyan/.conda/envs/meta/lib/python3.11/site-packages/nnsight/intervention/envoy.py", line 1048, in __getattr__
    raise AttributeError(f"{self} has no attribute {name}")
AttributeError: Lc0Model(
  (_lc0_model): GraphModule(
    (attn_body/transpose): OnnxTranspose()
    (initializers): Module()
    (attn_body/reshape): OnnxReshape()
    (attn_body/shape): OnnxShape()
    (attn_body/batch): OnnxSlice()
    (attn_body/pos_encoding_shape): OnnxConcat()
    (attn_body/expand): OnnxExpand()
    (attn_body/padded_input): OnnxConcat()
    (attn_body/reshape2): OnnxReshape()
    (attn_body/matmul)

In [10]:
# The model loads but when trying to use nnsight wrapper, there's an issue
# Let's try using the virtual environment in the repo
import subprocess
result = subprocess.run(['ls', '-la', '/net/scratch2/smallyan/leela_eval/.venv/bin/'], capture_output=True, text=True)
print(result.stdout)

FileNotFoundError: [Errno 2] No such file or directory: 'ls'

In [11]:
# Let me restart with proper imports after the install
import importlib

# Try to reimport
from leela_interp import Lc0sight, LeelaBoard
print("leela_interp imported")

# Check nnsight version
import nnsight
print(f"nnsight version: {nnsight.__version__}")

leela_interp imported
nnsight version: 0.5.2


In [12]:
# Now try loading the model again
os.chdir('/net/scratch2/smallyan/leela_eval')
device = "cuda" if torch.cuda.is_available() else "cpu"

try:
    model = Lc0sight("lc0-original.onnx", device=device)
    print("Model loaded successfully on", device)
    print(f"Model type: {type(model)}")
except Exception as e:
    print(f"ERROR: {type(e).__name__}: {str(e)[:500]}")

Using device: cuda


ERROR: AttributeError: Lc0Model(
  (_lc0_model): GraphModule(
    (attn_body/transpose): OnnxTranspose()
    (initializers): Module()
    (attn_body/reshape): OnnxReshape()
    (attn_body/shape): OnnxShape()
    (attn_body/batch): OnnxSlice()
    (attn_body/pos_encoding_shape): OnnxConcat()
    (attn_body/expand): OnnxExpand()
    (attn_body/padded_input): OnnxConcat()
    (attn_body/reshape2): OnnxReshape()
    (attn_body/matmul): OnnxMatMul()
    (attn_body/add): OnnxBinaryMathOperation()
    (attn_body/mish/softplu


In [13]:
import sys
print(f"Python executable: {sys.executable}")
print(f"Python version: {sys.version}")
print()
for p in sys.path[:5]:
    print(p)

Python executable: /home/smallyan/.conda/envs/meta/bin/python
Python version: 3.11.0 (main, Mar  1 2023, 18:26:19) [GCC 11.2.0]

/net/scratch2/smallyan/leela_eval/src
/home/smallyan/.conda/envs/meta/lib/python311.zip
/home/smallyan/.conda/envs/meta/lib/python3.11
/home/smallyan/.conda/envs/meta/lib/python3.11/lib-dynload



In [14]:
# Let's look at the actual error more closely - the issue is with how nnsight wraps the model
# The error is "has no attribute _envoy" which suggests API change in nnsight

# Try a different approach - create the model without nnsight wrapper first
from leela_interp.core.lc0 import Lc0Model

# Try loading just the base model
try:
    base_model = Lc0Model("lc0-original.onnx")
    base_model = base_model.to(device)
    print("Base model loaded successfully")
except Exception as e:
    print(f"Base model error: {e}")

Using device: cpu


Base model loaded successfully


In [15]:
# Check what attributes NNsight has in current version
from nnsight import NNsight
import inspect

# Check the init signature
print("NNsight init signature:")
print(inspect.signature(NNsight.__init__))
print()

# Check attributes
print("NNsight attributes:")
attrs = [a for a in dir(NNsight) if not a.startswith('__')]
print(attrs[:20])

NNsight init signature:
(self, *args, **kwargs)

NNsight attributes:
['_add_envoy', '_batch', '_handle_overloaded_mount', '_prepare_input', '_shallow_copy', '_update', '_update_alias', 'all', 'clear_edits', 'cpu', 'cuda', 'device', 'edit', 'export_edits', 'get', 'import_edits', 'input', 'inputs', 'interleave', 'interleaving']


In [16]:
# Try downgrading nnsight to a compatible version
# First check what version was expected
!pip index versions nnsight 2>/dev/null | head -5

/bin/bash: line 1: head: command not found
